# Week 0 Day 2: Advanced Data Cleaning

CariSurg MedTech Pathways, Healthcare AI track.

For Day 2, I cleaned the class columns from Tutorial 2 and then cleaned the Pulse column as my assigned column.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")


## Load the dataset

I loaded the same reduced emergency triage dataset from Day 1.


In [ ]:
candidate_paths = [
    Path("EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../data/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../source_materials/week0/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../../../source_materials/week0/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("/content/drive/MyDrive/ColabNotebooks/CariSurg_Triage_Test/EmergencyTriageDataset_Reduced_Dirty.csv"),
]

FILE_PATH = next((path for path in candidate_paths if path.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError("Upload the Week 0 CSV or update FILE_PATH.")

df = pd.read_csv(FILE_PATH)
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from {FILE_PATH}")
df.head()


## Start from the Day 1 Gender cleanup

I cleaned Gender first so the dataset stayed consistent with the Day 1 work.


In [ ]:
gender_map = {"male": 1, "female": 0, "1": 1, "0": 0}
df["Gender"] = df["Gender"].astype("string").str.strip().str.lower().map(gender_map).astype("Int64")
print(df["Gender"].value_counts(dropna=False).sort_index())


## Clean GCS

GCS measures level of consciousness. The valid range in the tutorial is 3 to 15. I converted the column to numbers, treated non-numeric values as missing, and used the median to fill missing values because most records are clustered at 15.


In [ ]:
df["GCS"] = pd.to_numeric(df["GCS"], errors="coerce")
invalid_gcs = (df["GCS"] < 3) | (df["GCS"] > 15)
print(f"GCS missing after numeric conversion: {df['GCS'].isna().sum()}")
print(f"GCS out-of-range values: {invalid_gcs.sum()}")

df.loc[invalid_gcs, "GCS"] = np.nan
gcs_median = df["GCS"].median()
df["GCS"] = df["GCS"].fillna(gcs_median)

print(f"GCS median used: {gcs_median}")
print(df["GCS"].describe())
print(f"GCS missing after cleaning: {df['GCS'].isna().sum()}")


## Clean SBP

SBP is systolic blood pressure. The valid range in the tutorial is 50 to 250 mmHg. I converted the values to numbers, replaced values outside that range with missing values, and filled them with the median.


In [ ]:
df["SBP"] = pd.to_numeric(df["SBP"], errors="coerce")
invalid_sbp = (df["SBP"] < 50) | (df["SBP"] > 250)
print(f"SBP missing after numeric conversion: {df['SBP'].isna().sum()}")
print(f"SBP out-of-range values: {invalid_sbp.sum()}")

df.loc[invalid_sbp, "SBP"] = np.nan
sbp_median = df["SBP"].median()
df["SBP"] = df["SBP"].fillna(sbp_median)

print(f"SBP median used: {sbp_median}")
print(df["SBP"].describe())
print(f"SBP missing after cleaning: {df['SBP'].isna().sum()}")


## Clean Temp

Temp had Celsius values, Celsius strings, and Fahrenheit strings. I converted everything to Celsius first. Then I used the tutorial range of 32.0 to 43.0 degrees Celsius and filled missing values with the median.


In [ ]:
def to_celsius(value):
    """Convert a temperature value to Celsius when it appears to be Fahrenheit.

    Values above 60 are treated as Fahrenheit. Missing values stay missing.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    try:
        if text.endswith("C"):
            return float(text[:-1])
        if text.endswith("F"):
            return (float(text[:-1]) - 32) * 5 / 9
        return float(text)
    except ValueError:
        return np.nan

df["Temp"] = df["Temp"].apply(to_celsius)
invalid_temp = (df["Temp"] < 32) | (df["Temp"] > 43)
print(f"Temp missing after conversion: {df['Temp'].isna().sum()}")
print(f"Temp out-of-range values: {invalid_temp.sum()}")

df.loc[invalid_temp, "Temp"] = np.nan
temp_median = round(df["Temp"].median(), 1)
df["Temp"] = df["Temp"].fillna(temp_median)

print(f"Temp median used: {temp_median}")
print(df["Temp"].describe())
print(f"Temp missing after cleaning: {df['Temp'].isna().sum()}")


## Clean Pulse

Pulse was my selected column. It records heart rate in beats per minute. The tutorial valid range is 20 to 250 bpm. I converted non-numeric entries to missing values, replaced impossible pulse values with missing values, and used the median because the column had error values and extreme outliers.


In [ ]:
print("Pulse values before cleaning:")
print(df["pulse"].value_counts(dropna=False).head(12))

df["pulse"] = pd.to_numeric(df["pulse"], errors="coerce")
invalid_pulse = (df["pulse"] < 20) | (df["pulse"] > 250)
print(f"Pulse missing after numeric conversion: {df['pulse'].isna().sum()}")
print(f"Pulse out-of-range values: {invalid_pulse.sum()}")

df.loc[invalid_pulse, "pulse"] = np.nan
pulse_median = df["pulse"].median()
df["pulse"] = df["pulse"].fillna(pulse_median)

print(f"Pulse median used: {pulse_median}")
print(df["pulse"].describe())
print(f"Pulse missing after cleaning: {df['pulse'].isna().sum()}")


## Final check

After cleaning, GCS, SBP, Temp, and Pulse had no missing values left. Pulse values were kept inside the valid range of 20 to 250 bpm.


In [ ]:
checks = df[["GCS", "SBP", "Temp", "pulse"]].isna().sum()
print("Missing values after cleaning:")
print(checks)
print(f"Pulse min after cleaning: {df['pulse'].min()}")
print(f"Pulse max after cleaning: {df['pulse'].max()}")
df[["ID", "GCS", "SBP", "Temp", "pulse"]].head(10)
